# GoldSight — MLflow + Optuna Full Pipeline

Pipeline ini mencakup:
- ✅ Baseline model (LR & RF default)
- ✅ Multiple experiments & runs
- ✅ Hyperparameter tuning dengan Optuna
- ✅ Tracking setiap Optuna trial ke MLflow
- ✅ Model Registry: tiap registrasi = versi baru
- ✅ Alias: `@champion`, `@candidate`, `@baseline`
- ✅ Run compare, run overview, run artifacts

## 1. Library

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
import json
import matplotlib
matplotlib.use('Agg')  # non-interactive backend agar bisa disimpan ke artifact
import matplotlib.pyplot as plt

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress verbose optuna

from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import xgboost as xgb

import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow import MlflowClient

warnings.filterwarnings('ignore')
print('Libraries loaded')

Libraries loaded


## 2. Load Data

In [2]:
BASE_DIR = r"D:\LAPORAN TUGAS\PBL"  # ← sesuaikan dengan foldermu

DATA_PATHS = {
    0.5: os.path.join(BASE_DIR, 'gold_historical_0_5g.csv'),
    1:   os.path.join(BASE_DIR, 'gold_historical_1g.csv'),
    2:   os.path.join(BASE_DIR, 'gold_historical_2g.csv'),
    3:   os.path.join(BASE_DIR, 'gold_historical_3g.csv'),
    5:   os.path.join(BASE_DIR, 'gold_historical_5g.csv'),
}

dfs = []
for weight, path in DATA_PATHS.items():
    df = pd.read_csv(path)
    df.columns = [col.lower().strip() for col in df.columns]
    df['updated_at'] = pd.to_datetime(df['updated_at'], errors='coerce')
    df['weight'] = weight
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f'Total rows: {len(df_all)}')
df_all.sort_values(['weight', 'updated_at']).groupby('weight').head(2)

Total rows: 405


,brand,resource,weight,sell_price,buyback_price,updated_at
0,ANTAM,galeri24,0.5,1721000,1374000,2026-02-02
1,ANTAM,galeri24,0.5,1620000,1361000,2026-02-03
81,ANTAM,galeri24,1.0,3330000,2748000,2026-02-02
82,ANTAM,galeri24,1.0,3129000,2722000,2026-02-03
162,ANTAM,galeri24,2.0,6594000,5496000,2026-02-02
163,ANTAM,galeri24,2.0,6191000,5445000,2026-02-03
243,ANTAM,galeri24,3.0,9863000,8244000,2026-02-02
244,ANTAM,galeri24,3.0,9259000,8168000,2026-02-03
324,ANTAM,galeri24,5.0,16401000,13740000,2026-02-02
325,ANTAM,galeri24,5.0,15395000,13613000,2026-02-03


In [3]:
# ── EDA gabungan: semua ukuran dalam satu laporan ───────────────────────────
# Berguna untuk melihat korelasi antar ukuran dan perbandingan distribusi

from ydata_profiling import ProfileReport
import os

df_combined = df_all.copy()
df_combined['updated_at'] = pd.to_datetime(df_combined['updated_at'])
df_combined['price_per_gram'] = df_combined['sell_price'] / df_combined['weight']
df_combined['spread'] = df_combined['sell_price'] - df_combined['buyback_price']
df_combined['spread_pct'] = (df_combined['spread'] / df_combined['sell_price'] * 100).round(2)

profile = ProfileReport(
    df_combined,
    title="GoldSight EDA — Semua Ukuran Emas (Combined)",
    dataset={
        "description": "Gabungan data historis harga emas ANTAM 0.5g-5g dari galeri24.",
    },
    variables={
        "descriptions": {
            "price_per_gram": "Harga per gram (sell_price / weight) — normalisasi antar ukuran",
            "spread":         "Selisih harga jual vs buyback (Rupiah)",
            "spread_pct":     "Spread dalam persen terhadap harga jual",
        }
    },
    correlations={
        "pearson":  {"calculate": True},
        "spearman": {"calculate": True},
        "kendall":  {"calculate": True},
    },
    explorative=True,
    progress_bar=True,
)

os.makedirs('./eda_reports', exist_ok=True)
out = './eda_reports/eda_combined.html'
profile.to_file(out)

print(f"Laporan EDA gabungan: {os.path.abspath(out)}")
profile.to_notebook_iframe()


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 9/9 [00:00<00:00, 1071.68it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Laporan EDA gabungan: d:\LAPORAN TUGAS\PBL\eda_reports\eda_combined.html


In [4]:
# ── Quick Stats — tampilkan ringkasan tanpa ydata-profiling ─────────────────

import pandas as pd

print("  RINGKASAN DATA — SEMUA UKURAN")
print(f"  Total baris  : {len(df_all):,}")
print(f"  Total kolom  : {len(df_all.columns)}")
print(f"  Kolom        : {df_all.columns.tolist()}")
print(f"  Missing vals : {df_all.isnull().sum().sum()}")
print()

for w in [0.5, 1, 2, 3, 5]:
    sub = df_all[df_all['weight'] == w]
    print(f"  {w}g  →  {len(sub)} baris | "
          f"sell_price: Rp {sub['sell_price'].min():,.0f} – Rp {sub['sell_price'].max():,.0f} | "
          f"mean: Rp {sub['sell_price'].mean():,.0f}")

print()
print("  STATISTIK DESKRIPTIF — sell_price per ukuran")
summary = df_all.groupby('weight')['sell_price'].describe()
print(summary.round(0).to_string())

print()
print("  RENTANG WAKTU DATA")
df_all['updated_at_dt'] = pd.to_datetime(df_all['updated_at'])
time_range = df_all.groupby('weight')['updated_at_dt'].agg(['min','max'])
print(time_range.to_string())


  RINGKASAN DATA — SEMUA UKURAN
  Total baris  : 405
  Total kolom  : 6
  Kolom        : ['brand', 'resource', 'weight', 'sell_price', 'buyback_price', 'updated_at']
  Missing vals : 0

  0.5g  →  81 baris | sell_price: Rp 1,505,000 – Rp 1,748,000 | mean: Rp 1,612,000
  1g  →  81 baris | sell_price: Rp 2,906,000 – Rp 3,386,000 | mean: Rp 3,117,309
  2g  →  81 baris | sell_price: Rp 5,749,000 – Rp 6,707,000 | mean: Rp 6,169,852
  3g  →  81 baris | sell_price: Rp 8,597,000 – Rp 10,034,000 | mean: Rp 9,228,074
  5g  →  81 baris | sell_price: Rp 14,294,000 – Rp 16,686,000 | mean: Rp 15,344,284

  STATISTIK DESKRIPTIF — sell_price per ukuran
        count        mean       std         min         25%         50%         75%         max
weight                                                                                         
0.5      81.0   1612000.0   74149.0   1505000.0   1539000.0   1604000.0   1680000.0   1748000.0
1.0      81.0   3117309.0  145662.0   2906000.0   2972000.0   31020

## 3. Cleaning & Feature Engineering

In [5]:
df = df_all[['updated_at', 'weight', 'sell_price']].copy()
df = df.rename(columns={'updated_at': 'date', 'sell_price': 'price'})
df = df.sort_values(['date', 'weight']).reset_index(drop=True)

df['price_per_gram'] = df['price'] / df['weight']

def create_features(group):
    group = group.sort_values('date')
    g = group['price_per_gram']
    # Lag features
    group['lag_1']  = g.shift(1)
    group['lag_3']  = g.shift(3)
    group['lag_7']  = g.shift(7)
    # Moving averages
    group['ma_7']   = g.shift(1).rolling(7).mean()
    group['ma_14']  = g.shift(1).rolling(14).mean()
    group['ma_21']  = g.shift(1).rolling(21).mean()
    # Volatility
    group['rolling_std_7'] = g.shift(1).rolling(7).std()
    # Momentum & Trend
    group['change']         = g.pct_change()
    group['momentum']       = g - g.shift(7)
    group['trend_strength'] = group['ma_7'] - group['ma_14']
    # Time
    group['day_of_week'] = group['date'].dt.dayofweek
    return group
df = df.groupby('weight', group_keys=False).apply(create_features)

# Targets
df['target_daily'] = df.groupby('weight')['price_per_gram'].transform(
    lambda x: x.shift(-1) - x)
df['target_weekly'] = df.groupby('weight')['price_per_gram'].transform(
    lambda x: x.shift(-7) - x)

df = df.dropna().reset_index(drop=True)
print(f'Shape after feature engineering: {df.shape}')
df.head(5)

Shape after feature engineering: (265, 17)


,date,weight,price,price_per_gram,lag_1,lag_3,lag_7,ma_7,ma_14,ma_21,rolling_std_7,change,momentum,trend_strength,day_of_week,target_daily,target_weekly
0,2026-02-23,0.5,1721000,3.442000e+06,3.424000e+06,3.350000e+06,3344000.0,3.350857e+06,3.350429e+06,3.342667e+06,55351.689440,0.005257,98000.000000,428.571429,0,44000.0,54000.0
1,2026-02-23,1.0,3331000,3.331000e+06,3.314000e+06,3.239000e+06,3234000.0,3.240714e+06,3.240286e+06,3.232429e+06,55355.560653,0.005130,97000.000000,428.571429,0,44000.0,55000.0
2,2026-02-23,2.0,6596000,3.298000e+06,3.280500e+06,3.205500e+06,3201000.0,3.207500e+06,3.206964e+06,3.199214e+06,55137.706397,0.005335,97000.000000,535.714286,0,44000.0,55500.0
3,2026-02-23,3.0,9866000,3.288667e+06,3.271333e+06,3.196333e+06,3192000.0,3.198286e+06,3.197786e+06,3.190048e+06,55203.807092,0.005299,96666.666667,500.000000,0,44000.0,56000.0
4,2026-02-23,5.0,16407000,3.281400e+06,3.263800e+06,3.189000e+06,3184600.0,3.190886e+06,3.190414e+06,3.182657e+06,55108.844679,0.005392,96800.000000,471.428571,0,44000.0,55800.0


## 4. Train / Val / Test Split

In [6]:
train_list, val_list, test_list = [], [], []

for w, group in df.groupby('weight'):
    group = group.sort_values('date')
    n = len(group)
    train_end = int(n * 0.70)
    val_end   = int(n * 0.85)
    train_list.append(group.iloc[:train_end])
    val_list.append(group.iloc[train_end:val_end])
    test_list.append(group.iloc[val_end:])

train_df = pd.concat(train_list).reset_index(drop=True)
val_df   = pd.concat(val_list).reset_index(drop=True)
test_df  = pd.concat(test_list).reset_index(drop=True)

FEATURES = ['weight','lag_1','lag_3','lag_7','ma_7','ma_14',
            'rolling_std_7','momentum','trend_strength','day_of_week']

print(f'Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}')

Train: (185, 17), Val: (40, 17), Test: (40, 17)


## 5. MLflow Setup

In [7]:
TRACKING_URI = 'file:./mlruns'
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient(tracking_uri=TRACKING_URI)

# Experiment names
EXP_BASELINE = 'GoldSight-Baseline'
EXP_TUNING   = 'GoldSight-Optuna-Tuning'
EXP_FINAL    = 'GoldSight-Final'

for exp_name in [EXP_BASELINE, EXP_TUNING]:
    if not mlflow.get_experiment_by_name(exp_name):
        mlflow.create_experiment(exp_name)

# Helper: safe evaluate
def evaluate(model, X, y):
    y_pred = model.predict(X)
    mae  = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2   = r2_score(y, y_pred)
    mape = np.mean(np.abs((y - y_pred) / (np.abs(y) + 1e-9))) * 100
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape}, y_pred

# Helper: save & log prediction plot
def log_prediction_plot(y_true, y_pred, title, filename='pred_vs_actual.png'):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(y_true.values[:100], label='Actual', alpha=0.8)
    ax.plot(y_pred[:100],        label='Predicted', alpha=0.8)
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(filename)
    plt.close(fig)
    mlflow.log_artifact(filename)
    os.remove(filename)

print('MLflow configured. Experiments created.')

MLflow configured. Experiments created.


## 6. Baseline Runs
> Experiment: `GoldSight-Baseline` — LinearRegression & RandomForest default, 2 horizons

In [8]:
mlflow.set_experiment(EXP_BASELINE)

BASELINE_MODELS = {
    'LinearRegression': LinearRegression(),
    'RandomForest_default': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost_default': xgb.XGBRegressor(n_estimators=100, random_state=42,
                                         verbosity=0, eval_metric='rmse')
}

baseline_registry = {}  # store run_id per horizon+model

for horizon in ['daily', 'weekly']:
    target = f'target_{horizon}'

    X_train, y_train = train_df[FEATURES], train_df[target]
    X_val,   y_val   = val_df[FEATURES],   val_df[target]
    X_test,  y_test  = test_df[FEATURES],  test_df[target]

    for model_name, model in BASELINE_MODELS.items():

        run_name = f'baseline_{model_name}_{horizon}'

        with mlflow.start_run(run_name=run_name) as run:

            # Tags
            mlflow.set_tags({
                'model_type': model_name,
                'horizon':    horizon,
                'stage':      'baseline',
                'dataset':    'GoldSight'
            })

            # Params
            mlflow.log_param('model_name', model_name)
            mlflow.log_param('horizon',    horizon)
            mlflow.log_param('n_features', len(FEATURES))
            mlflow.log_param('train_size', len(X_train))
            mlflow.log_param('test_size',  len(X_test))

            # Fit
            model.fit(X_train, y_train)

            # Val metrics
            val_metrics, _   = evaluate(model, X_val,  y_val)
            test_metrics, y_pred = evaluate(model, X_test, y_test)

            mlflow.log_metrics({f'val_{k}':  v for k, v in val_metrics.items()})
            mlflow.log_metrics({f'test_{k}': v for k, v in test_metrics.items()})

            # Artifact: prediction plot
            log_prediction_plot(y_test, y_pred,
                                f'Baseline {model_name} — {horizon}')

            # Artifact: feature importance (RF & XGB only)
            if hasattr(model, 'feature_importances_'):
                fi = pd.DataFrame({'feature': FEATURES,
                                   'importance': model.feature_importances_})\
                       .sort_values('importance', ascending=False)
                fi_path = f'feature_importance_{model_name}_{horizon}.csv'
                fi.to_csv(fi_path, index=False)
                mlflow.log_artifact(fi_path)
                os.remove(fi_path)

            # Log model
            if 'XGBoost' in model_name:
                mlflow.xgboost.log_model(model, artifact_path='model')
            else:
                mlflow.sklearn.log_model(model, artifact_path='model')

            baseline_registry[f'{model_name}_{horizon}'] = run.info.run_id
            print(f'{run_name} | RMSE={test_metrics["RMSE"]:,.0f}')

print('\nBaseline experiment selesai!')

2026/05/25 15:27:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/25 15:27:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


baseline_LinearRegression_daily | RMSE=24,982


2026/05/25 15:27:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/25 15:27:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


baseline_RandomForest_default_daily | RMSE=32,379


2026/05/25 15:27:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

baseline_LinearRegression_weekly | RMSE=67,286


2026/05/25 15:27:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/25 15:27:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


baseline_RandomForest_default_weekly | RMSE=62,030


2026/05/25 15:27:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


baseline_XGBoost_default_weekly | RMSE=60,545

Baseline experiment selesai!


## 7. Optuna Hyperparameter Tuning
> Experiment: `GoldSight-Optuna-Tuning` — setiap trial Optuna = 1 MLflow child-run

In [9]:
mlflow.set_experiment(EXP_TUNING)

best_tuned_models  = {}   # {horizon: (model, params, metrics)}
best_tuned_run_ids = {}   # {horizon: run_id}
all_studies        = {}

N_TRIALS = 20  # jumlah trial Optuna per horizon

for horizon in ['daily', 'weekly']:
    target = f'target_{horizon}'

    X_tr,  y_tr  = train_df[FEATURES], train_df[target]
    X_val, y_val = val_df[FEATURES],   val_df[target]
    X_te,  y_te  = test_df[FEATURES],  test_df[target]

    # ── PARENT RUN: satu parent per horizon ────────────────────────────────
    with mlflow.start_run(run_name=f'optuna_tuning_{horizon}') as parent_run:

        mlflow.set_tags({
            'horizon':  horizon,
            'stage':    'tuning',
            'n_trials': N_TRIALS
        })
        mlflow.log_param('horizon',  horizon)
        mlflow.log_param('n_trials', N_TRIALS)

        parent_run_id = parent_run.info.run_id

        # ── OPTUNA OBJECTIVE ───────────────────────────────────────────────
        def make_objective(X_tr, y_tr, X_val, y_val, horizon, parent_run_id):
            def objective(trial):
                model_choice = trial.suggest_categorical(
                    'model_type', ['RandomForest', 'XGBoost'])

                if model_choice == 'RandomForest':
                    params = {
                        'n_estimators':     trial.suggest_int('n_estimators', 50, 300),
                        'max_depth':        trial.suggest_int('max_depth', 3, 15),
                        'min_samples_split':trial.suggest_int('min_samples_split', 2, 10),
                        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 6),
                        'max_features':     trial.suggest_float('max_features', 0.3, 1.0),
                        'random_state': 42, 'n_jobs': -1
                    }
                    model = RandomForestRegressor(**params)
                else:
                    params = {
                        'n_estimators':  trial.suggest_int('n_estimators', 50, 300),
                        'max_depth':     trial.suggest_int('max_depth', 3, 10),
                        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                        'subsample':     trial.suggest_float('subsample', 0.5, 1.0),
                        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                        'reg_alpha':     trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
                        'reg_lambda':    trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
                        'random_state': 42, 'verbosity': 0, 'eval_metric': 'rmse'
                    }
                    model = xgb.XGBRegressor(**params)

                # ── CHILD RUN per trial ────────────────────────────────────
                with mlflow.start_run(run_name=f'trial_{trial.number}_{model_choice}',
                                      nested=True) as child_run:
                    mlflow.set_tag('trial_number', trial.number)
                    mlflow.set_tag('model_type',   model_choice)
                    mlflow.set_tag('horizon',      horizon)

                    # Log semua hyperparameter
                    mlflow.log_param('model_type', model_choice)
                    for k, v in params.items():
                        if k not in ('random_state','n_jobs','verbosity','eval_metric'):
                            mlflow.log_param(k, v)

                    model.fit(X_tr, y_tr)
                    metrics_val, _ = evaluate(model, X_val, y_val)

                    mlflow.log_metrics({f'val_{k}': v for k, v in metrics_val.items()})

                return metrics_val['RMSE']
            return objective

        study = optuna.create_study(
            direction='minimize',
            study_name=f'goldsight_{horizon}',
            sampler=optuna.samplers.TPESampler(seed=42)
        )
        study.optimize(
            make_objective(X_tr, y_tr, X_val, y_val, horizon, parent_run_id),
            n_trials=N_TRIALS,
            show_progress_bar=False
        )
        all_studies[horizon] = study

        # ── Refit best model on train+val ─────────────────────────────────
        best_params  = study.best_params.copy()
        model_choice = best_params.pop('model_type')

        X_full = pd.concat([X_tr, X_val])
        y_full = pd.concat([y_tr, y_val])

        if model_choice == 'RandomForest':
            best_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
        else:
            best_model = xgb.XGBRegressor(**best_params, random_state=42,
                                          verbosity=0, eval_metric='rmse')
        best_model.fit(X_full, y_full)

        test_metrics, y_pred = evaluate(best_model, X_te, y_te)

        # Log best trial summary to parent
        mlflow.log_param('best_model_type', model_choice)
        mlflow.log_params({f'best_{k}': v for k, v in best_params.items()})
        mlflow.log_metric('best_val_RMSE', study.best_value)
        mlflow.log_metrics({f'test_{k}': v for k, v in test_metrics.items()})

        # Artifact: Optuna optimization history
        trials_df = study.trials_dataframe()[['number','value','params_model_type']]
        trials_path = f'optuna_trials_{horizon}.csv'
        trials_df.to_csv(trials_path, index=False)
        mlflow.log_artifact(trials_path)
        os.remove(trials_path)

        # Artifact: prediction plot
        log_prediction_plot(y_te, y_pred,
                            f'Best Tuned {model_choice} — {horizon}')

        # Log best model
        if model_choice == 'XGBoost':
            mlflow.xgboost.log_model(best_model, artifact_path='model')
        else:
            mlflow.sklearn.log_model(best_model, artifact_path='model')

        best_tuned_models[horizon]  = (best_model, model_choice, best_params, test_metrics)
        best_tuned_run_ids[horizon] = parent_run_id

        print(f'[{horizon}] Best: {model_choice} | '
              f'RMSE={test_metrics["RMSE"]:,.0f} | R²={test_metrics["R2"]:.4f}')

print('\n Optuna tuning selesai!')

2026/05/25 15:27:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[daily] Best: XGBoost | RMSE=28,239 | R²=-0.1461


2026/05/25 15:27:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[weekly] Best: XGBoost | RMSE=44,560 | R²=-0.0725

 Optuna tuning selesai!


## 8. Final Experiment — Multiple Runs per Horizon
> Experiment: `GoldSight-Final` — run LR, RF, XGB + best-tuned untuk perbandingan

In [10]:
mlflow.set_experiment(EXP_FINAL)

final_run_ids = {}  # {horizon_model: run_id}

FINAL_CONFIGS = [
    ('LinearRegression',        lambda: LinearRegression()),
    ('RandomForest_tuned',      lambda: RandomForestRegressor(
                                    n_estimators=150, max_depth=8,
                                    random_state=42, n_jobs=-1)),
    ('XGBoost_tuned',           lambda: xgb.XGBRegressor(
                                    n_estimators=150, learning_rate=0.05,
                                    max_depth=6, subsample=0.8,
                                    random_state=42, verbosity=0)),
]

for horizon in ['daily', 'weekly']:
    target = f'target_{horizon}'

    X_train, y_train = train_df[FEATURES], train_df[target]
    X_val,   y_val   = val_df[FEATURES],   val_df[target]
    X_test,  y_test  = test_df[FEATURES],  test_df[target]

    for model_label, model_fn in FINAL_CONFIGS:
        model = model_fn()
        run_name = f'final_{model_label}_{horizon}'

        with mlflow.start_run(run_name=run_name) as run:
            mlflow.set_tags({
                'model_type': model_label,
                'horizon':    horizon,
                'stage':      'final'
            })
            mlflow.log_param('model_name', model_label)
            mlflow.log_param('horizon',    horizon)

            # Log hyperparams
            params = {k: v for k, v in model.get_params().items()
                      if v is not None and k not in ('verbosity','eval_metric')}
            mlflow.log_params(params)

            model.fit(X_train, y_train)

            val_m, _         = evaluate(model, X_val, y_val)
            test_m, y_pred   = evaluate(model, X_test, y_test)

            mlflow.log_metrics({f'val_{k}':  v for k, v in val_m.items()})
            mlflow.log_metrics({f'test_{k}': v for k, v in test_m.items()})

            log_prediction_plot(y_test, y_pred,
                                f'Final {model_label} — {horizon}')

            if hasattr(model, 'feature_importances_'):
                fi = pd.DataFrame({'feature': FEATURES,
                                   'importance': model.feature_importances_})\
                       .sort_values('importance', ascending=False)
                fi_path = f'fi_{model_label}_{horizon}.csv'
                fi.to_csv(fi_path, index=False)
                mlflow.log_artifact(fi_path)
                os.remove(fi_path)

            if 'XGBoost' in model_label:
                mlflow.xgboost.log_model(model, artifact_path='model')
            else:
                mlflow.sklearn.log_model(model, artifact_path='model')

            final_run_ids[f'{model_label}_{horizon}'] = run.info.run_id
            print(f'{run_name} | RMSE={test_m["RMSE"]:,.0f}')

print('\n Final experiment selesai!')

2026/05/25 15:28:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/25 15:28:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


final_LinearRegression_daily | RMSE=24,982


2026/05/25 15:28:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/25 15:28:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


final_RandomForest_tuned_daily | RMSE=30,401


2026/05/25 15:28:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

final_LinearRegression_weekly | RMSE=67,286


2026/05/25 15:28:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/25 15:28:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


final_RandomForest_tuned_weekly | RMSE=64,169


2026/05/25 15:28:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


final_XGBoost_tuned_weekly | RMSE=57,326

 Final experiment selesai!


In [11]:
# GoldSight — Log Optuna Best Model into GoldSight-Final
import os
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Pastikan tracking URI sama
TRACKING_URI = "file:./mlruns"
mlflow.set_tracking_uri(TRACKING_URI)

# Experiment final
EXP_FINAL = "GoldSight-Final"
mlflow.set_experiment(EXP_FINAL)


# ------------------------------------------------------------
# Helper evaluasi
# ------------------------------------------------------------

def evaluate_final_model(model, X, y):
    y_pred = model.predict(X)

    mae = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)
    mape = np.mean(np.abs((y - y_pred) / (np.abs(y) + 1e-9))) * 100

    metrics = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape
    }

    return metrics, y_pred


# ------------------------------------------------------------
# Helper log prediction plot
# ------------------------------------------------------------

def log_final_prediction_plot(y_true, y_pred, title, filename):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(y_true.values[:100], label="Actual", alpha=0.8)
    ax.plot(y_pred[:100], label="Predicted", alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("Sample")
    ax.set_ylabel("Target Delta Price")
    ax.legend()
    fig.tight_layout()

    fig.savefig(filename, dpi=120)
    plt.close(fig)

    mlflow.log_artifact(filename)

    if os.path.exists(filename):
        os.remove(filename)


# ------------------------------------------------------------
# Log Optuna best model ke GoldSight-Final
# ------------------------------------------------------------

final_optuna_run_ids = {}

for horizon in ["daily", "weekly"]:

    if horizon not in best_tuned_models:
        print(f"[SKIP] Tidak ada best_tuned_models untuk horizon: {horizon}")
        continue

    target = f"target_{horizon}"

    X_test = test_df[FEATURES]
    y_test = test_df[target]

    # Ambil model terbaik hasil Optuna
    best_model, model_choice, best_params, old_test_metrics = best_tuned_models[horizon]

    # Nama model pendek
    if model_choice == "XGBoost":
        model_short = "XGB"
    elif model_choice == "RandomForest":
        model_short = "RF"
    else:
        model_short = model_choice

    # Nama run final yang jelas
    run_name = f"{horizon}_{model_short}_optuna_final"

    with mlflow.start_run(run_name=run_name) as run:

        # Tags
        mlflow.set_tags({
            "stage": "final",
            "horizon": horizon,
            "model_type": model_choice,
            "model_short": model_short,
            "tuning_type": "optuna",
            "selection_scope": "final_experiment",
            "source_experiment": "GoldSight-Optuna-Tuning",
            "description": "Best Optuna model re-logged into GoldSight-Final"
        })

        # Params
        mlflow.log_param("model_name", model_choice)
        mlflow.log_param("horizon", horizon)
        mlflow.log_param("tuning_type", "optuna")
        mlflow.log_param("n_features", len(FEATURES))
        mlflow.log_param("train_size", len(train_df))
        mlflow.log_param("val_size", len(val_df))
        mlflow.log_param("test_size", len(test_df))

        # Log best params dari Optuna
        if isinstance(best_params, dict):
            for k, v in best_params.items():
                mlflow.log_param(f"best_{k}", v)

        # Evaluasi ulang di test set
        test_metrics, y_pred = evaluate_final_model(best_model, X_test, y_test)

        mlflow.log_metrics({
            "test_MAE": test_metrics["MAE"],
            "test_RMSE": test_metrics["RMSE"],
            "test_R2": test_metrics["R2"],
            "test_MAPE": test_metrics["MAPE"]
        })

        # Log prediction plot
        plot_file = f"prediction_plot_{horizon}_{model_short}_optuna_final.png"

        log_final_prediction_plot(
            y_true=y_test,
            y_pred=y_pred,
            title=f"Final Optuna {model_choice} — {horizon}",
            filename=plot_file
        )

        # Log feature importance jika tersedia
        if hasattr(best_model, "feature_importances_"):
            fi = pd.DataFrame({
                "feature": FEATURES,
                "importance": best_model.feature_importances_
            }).sort_values("importance", ascending=False)

            fi_path = f"feature_importance_{horizon}_{model_short}_optuna_final.csv"
            fi.to_csv(fi_path, index=False)
            mlflow.log_artifact(fi_path)

            if os.path.exists(fi_path):
                os.remove(fi_path)

        # Log model
        if model_choice == "XGBoost":
            mlflow.xgboost.log_model(best_model, artifact_path="model")
        else:
            mlflow.sklearn.log_model(best_model, artifact_path="model")

        final_optuna_run_ids[horizon] = run.info.run_id

        print(
            f"[OK] {run_name} logged to GoldSight-Final | "
            f"RMSE={test_metrics['RMSE']:,.0f} | "
            f"MAE={test_metrics['MAE']:,.0f} | "
            f"R2={test_metrics['R2']:.4f}"
        )

print("\nSelesai. Best Optuna model sudah dicatat ulang ke GoldSight-Final.")

2026/05/25 15:28:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[OK] daily_XGB_optuna_final logged to GoldSight-Final | RMSE=28,239 | MAE=24,217 | R2=-0.1461


2026/05/25 15:28:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[OK] weekly_XGB_optuna_final logged to GoldSight-Final | RMSE=44,560 | MAE=39,162 | R2=-0.0725

Selesai. Best Optuna model sudah dicatat ulang ke GoldSight-Final.


## 9. Runs Result Summary & Compare
> Mengambil semua run dari tiga experiment untuk dibandingkan

In [12]:
def load_experiment_runs(exp_name):
    exp = mlflow.get_experiment_by_name(exp_name)
    if exp is None:
        return pd.DataFrame()
    runs = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="tags.mlflow.runName != ''"  # skip nested child runs
    )
    return runs

all_runs = []
for exp_name in [EXP_BASELINE, EXP_TUNING, EXP_FINAL]:
    runs = load_experiment_runs(exp_name)
    if not runs.empty:
        runs['experiment'] = exp_name
        all_runs.append(runs)

compare_df = pd.concat(all_runs, ignore_index=True)

# Select display columns
cols = ['run_id','tags.mlflow.runName','tags.horizon','tags.stage',
        'metrics.test_RMSE','metrics.test_MAE','metrics.test_R2','experiment']
cols_exist = [c for c in cols if c in compare_df.columns]
display_df = compare_df[cols_exist].copy()
display_df.columns = [c.replace('tags.','').replace('metrics.','') for c in display_df.columns]
display_df = display_df.sort_values('test_RMSE')

print(f'Total runs: {len(display_df)}')
display_df.head(20)

Total runs: 546


,run_id,mlflow.runName,horizon,stage,test_RMSE,test_MAE,test_R2,experiment
5,61cfa899e91a433b93d08eeb24c63afb,baseline_LinearRegression_daily,daily,baseline,24982.244678,19989.120084,0.103042,GoldSight-Baseline
11,f2827bd45a8a41a0be5a274648303b76,baseline_LinearRegression_daily,daily,baseline,24982.244678,19989.120084,0.103042,GoldSight-Baseline
23,6ac4637521194d39961e66eba9b80f84,baseline_LinearRegression_daily,daily,baseline,24982.244678,19989.120084,0.103042,GoldSight-Baseline
17,ab05d2d90a9845b7926af6cbdfd1bdbf,baseline_LinearRegression_daily,daily,baseline,24982.244678,19989.120084,0.103042,GoldSight-Baseline
29,2fae4c26555b4af39ea7783c63647393,baseline_LinearRegression_daily,daily,baseline,24982.244678,19989.120084,0.103042,GoldSight-Baseline
47,db5ee7110bf4434788bab132158109eb,baseline_LinearRegression_daily,daily,baseline,24982.244678,19989.120084,0.103042,GoldSight-Baseline
41,d9231fe88bbe4770b17208821991000f,baseline_LinearRegression_daily,daily,baseline,24982.244678,19989.120084,0.103042,GoldSight-Baseline
35,2f4190b5fba94701bfcc1e317b619ba6,baseline_LinearRegression_daily,daily,baseline,24982.244678,19989.120084,0.103042,GoldSight-Baseline
53,4fd278e37e224a5e81a50a3f419e00bd,baseline_LinearRegression_daily,daily,baseline,24982.244678,19989.120084,0.103042,GoldSight-Baseline
59,840c0472abce45ab93142ad63480962e,baseline_LinearRegression_daily,daily,baseline,24982.244678,19989.120084,0.103042,GoldSight-Baseline


In [13]:
# ── Best model per horizon ──────────────────────────────────────────────────
best_per_horizon = {}

for horizon in ['daily', 'weekly']:
    sub = display_df[display_df['horizon'] == horizon].dropna(subset=['test_RMSE'])
    if not sub.empty:
        best_row = sub.loc[sub['test_RMSE'].idxmin()]
        best_per_horizon[horizon] = best_row
        print(f'[{horizon.upper()}] Best → {best_row["mlflow.runName"]} '
              f'| RMSE={best_row["test_RMSE"]:,.0f} | R²={best_row["test_R2"]:.4f}')

[DAILY] Best → baseline_LinearRegression_daily | RMSE=24,982 | R²=0.1030
[WEEKLY] Best → optuna_tuning_weekly | RMSE=44,560 | R²=-0.0725


## 10. Model Registry
> Setiap registrasi = versi baru. Alias: `@champion` (terbaik), `@candidate` (baru), `@baseline`

In [14]:
# ── Registry helper functions ───────────────────────────────────────────────

def register_model_version(run_id, model_artifact_path, registered_name, description=''):
    """Register model from a run. Returns the new ModelVersion."""
    model_uri = f'runs:/{run_id}/{model_artifact_path}'
    mv = mlflow.register_model(model_uri=model_uri, name=registered_name)
    if description:
        client.update_model_version(
            name=registered_name,
            version=mv.version,
            description=description
        )
    print(f'Registered [{registered_name}] v{mv.version}')
    return mv

def set_alias(registered_name, version, alias):
    """Assign alias to a model version."""
    client.set_registered_model_alias(
        name=registered_name,
        alias=alias,
        version=str(version)
    )
    print(f'Alias @{alias} → [{registered_name}] v{version}')

print('Registry helpers ready.')

Registry helpers ready.


In [15]:
# ── 1. Register BASELINE models ─────────────────────────────────────────────
print('=== Registering BASELINE models ===')

for horizon in ['daily', 'weekly']:

    # LinearRegression baseline
    lr_key     = f'LinearRegression_{horizon}'
    lr_run_id  = baseline_registry.get(lr_key)
    reg_name   = f'GoldSight-{horizon.capitalize()}-Predictor'

    if lr_run_id:
        mv_lr = register_model_version(
            run_id=lr_run_id,
            model_artifact_path='model',
            registered_name=reg_name,
            description=f'Baseline LinearRegression for {horizon} prediction'
        )
        set_alias(reg_name, mv_lr.version, 'baseline')

    # RandomForest baseline
    rf_key    = f'RandomForest_default_{horizon}'
    rf_run_id = baseline_registry.get(rf_key)

    if rf_run_id:
        mv_rf = register_model_version(
            run_id=rf_run_id,
            model_artifact_path='model',
            registered_name=reg_name,
            description=f'Baseline RandomForest (default params) for {horizon}'
        )

print()

=== Registering BASELINE models ===


Registered model 'GoldSight-Daily-Predictor' already exists. Creating a new version of this model...
2026/05/25 15:28:55 WARNING mlflow.tracking._model_registry.fluent: Run with id 61cfa899e91a433b93d08eeb24c63afb has no artifacts at artifact path 'model', registering model based on models:/m-77a69db68da4444a9c424a4f237cf59a instead
Created version '37' of model 'GoldSight-Daily-Predictor'.


Registered [GoldSight-Daily-Predictor] v37
Alias @baseline → [GoldSight-Daily-Predictor] v37


Registered model 'GoldSight-Daily-Predictor' already exists. Creating a new version of this model...
2026/05/25 15:28:57 WARNING mlflow.tracking._model_registry.fluent: Run with id 5afc002fe0b34d62bbae53559ad46a29 has no artifacts at artifact path 'model', registering model based on models:/m-2764fcd50b6e4e43b77fc691a1b8481f instead
Created version '38' of model 'GoldSight-Daily-Predictor'.


Registered [GoldSight-Daily-Predictor] v38


Registered model 'GoldSight-Weekly-Predictor' already exists. Creating a new version of this model...
2026/05/25 15:28:59 WARNING mlflow.tracking._model_registry.fluent: Run with id 487213fdb361436898f0793624a3dfc4 has no artifacts at artifact path 'model', registering model based on models:/m-3f223b9c10714433ae92ddabf6d95470 instead
Created version '37' of model 'GoldSight-Weekly-Predictor'.


Registered [GoldSight-Weekly-Predictor] v37
Alias @baseline → [GoldSight-Weekly-Predictor] v37


Registered model 'GoldSight-Weekly-Predictor' already exists. Creating a new version of this model...
2026/05/25 15:29:01 WARNING mlflow.tracking._model_registry.fluent: Run with id 9b64e21b198c480d96d247e7266493f9 has no artifacts at artifact path 'model', registering model based on models:/m-9236f80100e2485fad4c39725ef19695 instead
Created version '38' of model 'GoldSight-Weekly-Predictor'.


Registered [GoldSight-Weekly-Predictor] v38



In [16]:
# ── 2. Register TUNED models sebagai @candidate ──────────────────────────────
print('=== Registering TUNED models as @candidate ===')

for horizon in ['daily', 'weekly']:
    run_id   = best_tuned_run_ids.get(horizon)
    reg_name = f'GoldSight-{horizon.capitalize()}-Predictor'

    if run_id:
        mv = register_model_version(
            run_id=run_id,
            model_artifact_path='model',
            registered_name=reg_name,
            description=f'Optuna-tuned best model for {horizon}. Candidate for champion.'
        )
        set_alias(reg_name, mv.version, 'candidate')

print()

=== Registering TUNED models as @candidate ===


Registered model 'GoldSight-Daily-Predictor' already exists. Creating a new version of this model...
2026/05/25 15:29:03 WARNING mlflow.tracking._model_registry.fluent: Run with id 157c2c42c58d4082b2fa5469d5d7392d has no artifacts at artifact path 'model', registering model based on models:/m-be65fc4bef1c4a80a9ecf0c5d740866e instead
Created version '39' of model 'GoldSight-Daily-Predictor'.


Registered [GoldSight-Daily-Predictor] v39
Alias @candidate → [GoldSight-Daily-Predictor] v39


Registered model 'GoldSight-Weekly-Predictor' already exists. Creating a new version of this model...
2026/05/25 15:29:04 WARNING mlflow.tracking._model_registry.fluent: Run with id 1071804243d34b8a87f0d0000c855c9e has no artifacts at artifact path 'model', registering model based on models:/m-d874c3eb8c4d4db18f84f5d86a38b234 instead
Created version '39' of model 'GoldSight-Weekly-Predictor'.


Registered [GoldSight-Weekly-Predictor] v39
Alias @candidate → [GoldSight-Weekly-Predictor] v39



In [17]:
# ── 3. Register FINAL models & set @champion ─────────────────────────────────
print('Registering FINAL models — setting @champion')

for horizon in ['daily', 'weekly']:
    reg_name = f'GoldSight-{horizon.capitalize()}-Predictor'

    best_row = best_per_horizon.get(horizon)
    if best_row is None:
        continue

    best_run_id_str = best_row['run_id']

    # Cari run_id dari final_run_ids
    champion_run_id = None
    for key, rid in final_run_ids.items():
        if horizon in key:
            # Prefer yang RMSE terbaik (sudah dihitung di best_per_horizon)
            if rid == best_run_id_str:
                champion_run_id = rid
                break

    # Fallback: gunakan run_id langsung dari compare
    if champion_run_id is None:
        champion_run_id = best_run_id_str

    mv = register_model_version(
        run_id=champion_run_id,
        model_artifact_path='model',
        registered_name=reg_name,
        description=f'Champion model for {horizon} — lowest RMSE in final experiment'
    )
    set_alias(reg_name, mv.version, 'champion')

print()
print('Model Registry selesai!')

Registering FINAL models — setting @champion


Registered model 'GoldSight-Daily-Predictor' already exists. Creating a new version of this model...
2026/05/25 15:29:07 WARNING mlflow.tracking._model_registry.fluent: Run with id 61cfa899e91a433b93d08eeb24c63afb has no artifacts at artifact path 'model', registering model based on models:/m-77a69db68da4444a9c424a4f237cf59a instead
Created version '40' of model 'GoldSight-Daily-Predictor'.


Registered [GoldSight-Daily-Predictor] v40
Alias @champion → [GoldSight-Daily-Predictor] v40


Registered model 'GoldSight-Weekly-Predictor' already exists. Creating a new version of this model...
2026/05/25 15:29:09 WARNING mlflow.tracking._model_registry.fluent: Run with id ebb13a0aa80e42f185acdf0370748c45 has no artifacts at artifact path 'model', registering model based on models:/m-c36db3d47b61439da7947c2bc9282580 instead
Created version '40' of model 'GoldSight-Weekly-Predictor'.


Registered [GoldSight-Weekly-Predictor] v40
Alias @champion → [GoldSight-Weekly-Predictor] v40

Model Registry selesai!


## 11. Registry Overview: Versi & Alias

In [18]:
print('MODEL REGISTRY OVERVIEW')

for horizon in ['daily', 'weekly']:
    reg_name = f'GoldSight-{horizon.capitalize()}-Predictor'
    print(f'\n{reg_name}')
    try:
        versions = client.search_model_versions(f"name='{reg_name}'")
        for v in sorted(versions, key=lambda x: int(x.version)):
            aliases = v.aliases if hasattr(v, 'aliases') else []
            alias_str = ', '.join([f'@{a}' for a in aliases]) if aliases else ''
            print(f'  v{v.version} | {alias_str:20s} | {v.description[:60]}')
    except Exception as e:
        print(f'  [info] {e}')

MODEL REGISTRY OVERVIEW

GoldSight-Daily-Predictor
  v1 |                      | Baseline LinearRegression for daily prediction
  v2 |                      | Baseline RandomForest (default params) for daily
  v3 |                      | Optuna-tuned best model for daily. Candidate for champion.
  v4 |                      | Champion model for daily — lowest RMSE in final experiment
  v5 |                      | Baseline LinearRegression for daily prediction
  v6 |                      | Baseline RandomForest (default params) for daily
  v7 |                      | Optuna-tuned best model for daily. Candidate for champion.
  v8 |                      | Champion model for daily — lowest RMSE in final experiment
  v9 |                      | Baseline LinearRegression for daily prediction
  v10 |                      | Baseline RandomForest (default params) for daily
  v11 |                      | Optuna-tuned best model for daily. Candidate for champion.
  v12 |                      | Cha

## 12. Load & Predict menggunakan Alias
> Demo load model dari registry dengan alias `@champion`

In [19]:
for horizon in ['daily', 'weekly']:
    reg_name = f'GoldSight-{horizon.capitalize()}-Predictor'
    target   = f'target_{horizon}'
    X_test, y_test = test_df[FEATURES], test_df[target]

    for alias in ['champion', 'candidate', 'baseline']:
        try:
            model_uri  = f'models:/{reg_name}@{alias}'
            loaded_mdl = mlflow.pyfunc.load_model(model_uri)
            y_pred     = loaded_mdl.predict(X_test)
            rmse       = np.sqrt(mean_squared_error(y_test, y_pred))
            mae        = mean_absolute_error(y_test, y_pred)
            print(f'[{horizon}] @{alias:10s} → RMSE={rmse:,.0f} | MAE={mae:,.0f}')
        except Exception as e:
            print(f'[{horizon}] @{alias:10s} → (not set) {e}')

[daily] @champion   → RMSE=24,982 | MAE=19,989
[daily] @candidate  → RMSE=28,239 | MAE=24,217
[daily] @baseline   → RMSE=24,982 | MAE=19,989
[weekly] @champion   → RMSE=44,560 | MAE=39,162
[weekly] @candidate  → RMSE=44,560 | MAE=39,162
[weekly] @baseline   → RMSE=67,286 | MAE=55,162


In [20]:
# ── Definisi Rule Engine ─────────────────────────────────────────────────────
EXP_RULES = 'GoldSight-Rules'
if not mlflow.get_experiment_by_name(EXP_RULES):
    mlflow.create_experiment(EXP_RULES)

def generate_recommendation(sell_price, predicted_price, buyback_price=None,
                             horizon='daily'):
    """
    Rule-based engine: BUY / HOLD / SELL

    Rules:
      delta_pct > +1.5%          → BUY   (momentum naik signifikan)
      delta_pct < -1.5%          → SELL  (momentum turun signifikan)
      -0.5% <= delta_pct <= 0.5% → HOLD  (harga stabil)
      selain itu                 → HOLD  (pergerakan moderat)
    """
    delta     = predicted_price - sell_price
    delta_pct = (delta / sell_price) * 100 if sell_price > 0 else 0
    spread    = ((sell_price - buyback_price) / sell_price * 100
                 if buyback_price else None)
    label     = 'harian' if horizon == 'daily' else 'mingguan'

    if delta_pct > 1.5:
        rtype  = 'BUY'
        reason = (
            f"Prediksi {label} menunjukkan kenaikan Rp {abs(delta):,.0f}/gram "
            f"({delta_pct:+.2f}%). Momentum naik → sinyal BELI."
        )
        if spread and spread > 5:
            reason += f" Spread beli-jual {spread:.1f}% — pertimbangkan timing."

    elif delta_pct < -1.5:
        rtype  = 'SELL'
        reason = (
            f"Prediksi {label} menunjukkan penurunan Rp {abs(delta):,.0f}/gram "
            f"({delta_pct:+.2f}%). Momentum turun → disarankan JUAL."
        )

    elif -0.5 <= delta_pct <= 0.5:
        rtype  = 'HOLD'
        reason = (
            f"Prediksi {label} menunjukkan pergerakan minimal ({delta_pct:+.2f}%). "
            f"Harga cenderung stabil → TAHAN posisi."
        )

    else:
        arah   = 'naik' if delta_pct > 0 else 'turun'
        rtype  = 'HOLD'
        reason = (
            f"Prediksi {label} harga {arah} moderat Rp {abs(delta):,.0f}/gram "
            f"({delta_pct:+.2f}%). Belum ada sinyal kuat → TAHAN."
        )

    return {
        'recommendation': rtype,
        'reason':         reason,
        'delta':          round(delta, 2),
        'delta_pct':      round(delta_pct, 2),
        'spread_pct':     round(spread, 2) if spread else None,
    }

print('Rule engine siap: BUY / HOLD / SELL')

Rule engine siap: BUY / HOLD / SELL


In [21]:
# ── Integrasi MA + Prediksi + Rule, di-log ke MLflow ─────────────────────────

def ma_signal(ma_7, ma_14):
    """Sinyal dari perbandingan MA7 vs MA14."""
    if pd.isna(ma_7) or pd.isna(ma_14):
        return 'neutral', 'MA tidak tersedia'
    if ma_7 > ma_14:
        return 'bullish', f'Golden cross: MA7 ({ma_7:,.0f}) > MA14 ({ma_14:,.0f})'
    elif ma_7 < ma_14:
        return 'bearish', f'Death cross: MA7 ({ma_7:,.0f}) < MA14 ({ma_14:,.0f})'
    return 'neutral', f'MA7 = MA14 = {ma_7:,.0f}'


def enrich_recommendation(rec_data, ma_sig):
    """Perkuat rekomendasi berdasarkan sinyal MA."""
    rec   = rec_data['recommendation']
    delta = rec_data['delta_pct']

    if ma_sig == 'bullish' and rec == 'HOLD' and delta > 0:
        rec_data['recommendation'] = 'BUY'
        rec_data['reason'] += f' Diperkuat golden cross MA.'
    elif ma_sig == 'bearish' and rec == 'HOLD' and delta < 0:
        rec_data['recommendation'] = 'SELL'
        rec_data['reason'] += f' Diperkuat death cross MA.'
    return rec_data


print('MA signal engine siap')


MA signal engine siap


In [22]:
# ── Jalankan Rules pada Test Set & Log ke MLflow ─────────────────────────────

mlflow.set_experiment(EXP_RULES)

REC_COLORS = {'BUY': '🟢', 'HOLD': '🟡', 'SELL': '🔴'}
all_rec_results = []

for horizon in ['daily', 'weekly']:
    target   = f'target_{horizon}'
    X_test   = test_df[FEATURES]
    y_test   = test_df[target]
    y_actual = test_df['price_per_gram'] if 'price_per_gram' in test_df.columns else None

    # Ambil model terbaik (dari best_tuned_models jika ada, else fallback)
    if horizon in best_tuned_models:
        best_model, model_name, best_params, _ = best_tuned_models[horizon]
    else:
        best_model = list(BASELINE_MODELS.values())[0]
        model_name = 'LinearRegression'
        best_params = {}

    run_name = f'rules_{horizon}'

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({
            'stage':   'rule_engine',
            'horizon': horizon, 
        })
        mlflow.log_param('horizon',    horizon)
        mlflow.log_param('model_used', model_name)

        # Prediksi delta
        y_pred_delta = best_model.predict(X_test)

        # Hitung harga prediksi dari delta
        base_prices = test_df['price_per_gram'].values if 'price_per_gram' in test_df.columns                       else (test_df[FEATURES]['lag_1'].values)

        rec_counts = {'BUY': 0, 'HOLD': 0, 'SELL': 0}
        sample_recs = []

        for i, (delta_pred, base_price) in enumerate(zip(y_pred_delta, base_prices)):
            pred_price = base_price + delta_pred

            # MA signal dari fitur test set
            row      = test_df.iloc[i]
            ma7_val  = row.get('ma_7',  np.nan) if 'ma_7'  in test_df.columns else np.nan
            ma14_val = row.get('ma_14', np.nan) if 'ma_14' in test_df.columns else np.nan
            ma_sig, ma_reason = ma_signal(ma7_val, ma14_val)

            # Generate rekomendasi
            rec_data = generate_recommendation(
                sell_price      = float(base_price),
                predicted_price = float(pred_price),
                horizon         = horizon
            )
            rec_data = enrich_recommendation(rec_data, ma_sig)

            rec_counts[rec_data['recommendation']] += 1

            # Simpan sample untuk display
            if i < 5:
                sample_recs.append({
                    'idx':            i,
                    'base_price':     round(base_price),
                    'pred_price':     round(pred_price),
                    'delta_pct':      rec_data['delta_pct'],
                    'ma_signal':      ma_sig,
                    'recommendation': rec_data['recommendation'],
                    'reason':         rec_data['reason'][:80],
                })

        total = len(y_pred_delta)
        buy_pct  = rec_counts['BUY']  / total * 100
        hold_pct = rec_counts['HOLD'] / total * 100
        sell_pct = rec_counts['SELL'] / total * 100

        # Log distribusi rekomendasi
        mlflow.log_metrics({
            'rec_BUY_count':  rec_counts['BUY'],
            'rec_HOLD_count': rec_counts['HOLD'],
            'rec_SELL_count': rec_counts['SELL'],
            'rec_BUY_pct':    round(buy_pct, 2),
            'rec_HOLD_pct':   round(hold_pct, 2),
            'rec_SELL_pct':   round(sell_pct, 2),
            'total_samples':  total,
        })

        # Artifact: rekap distribusi CSV
        dist_df = pd.DataFrame([{
            'horizon':    horizon,
            'BUY_count':  rec_counts['BUY'],
            'HOLD_count': rec_counts['HOLD'],
            'SELL_count': rec_counts['SELL'],
            'BUY_pct':    round(buy_pct, 2),
            'HOLD_pct':   round(hold_pct, 2),
            'SELL_pct':   round(sell_pct, 2),
        }])
        dist_path = f'rec_distribution_{horizon}.csv'
        dist_df.to_csv(dist_path, index=False)
        mlflow.log_artifact(dist_path)
        os.remove(dist_path)

        # Artifact: sample rekomendasi CSV
        sample_df = pd.DataFrame(sample_recs)
        if not sample_df.empty:
            samp_path = f'rec_samples_{horizon}.csv'
            sample_df.to_csv(samp_path, index=False)
            mlflow.log_artifact(samp_path)
            os.remove(samp_path)

        all_rec_results.append({
            'horizon':   horizon,
            'counts':    rec_counts,
            'buy_pct':   buy_pct,
            'hold_pct':  hold_pct,
            'sell_pct':  sell_pct,
            'run_id':    run.info.run_id,
        })

        print(f'  [{horizon.upper()}] BUY={rec_counts["BUY"]} ({buy_pct:.0f}%) | '
              f'HOLD={rec_counts["HOLD"]} ({hold_pct:.0f}%) | '
              f'SELL={rec_counts["SELL"]} ({sell_pct:.0f}%)')
        print(f'  Run ID: {run.info.run_id[:16]}...')

print('\nRule engine selesai & tersimpan di MLflow!')


  [DAILY] BUY=16 (40%) | HOLD=19 (48%) | SELL=5 (12%)
  Run ID: 3b71e411af6042b5...
  [WEEKLY] BUY=4 (10%) | HOLD=26 (65%) | SELL=10 (25%)
  Run ID: f450ef2f04fa4c47...

Rule engine selesai & tersimpan di MLflow!


In [23]:
# ── Display hasil rekomendasi ─────────────────────────────────────────────────
print('  DISTRIBUSI REKOMENDASI PADA TEST SET')

for res in all_rec_results:
    h = res['horizon'].upper()
    print(f'\n  📅 Horizon: {h}')
    print(f'  🟢 BUY  : {res["counts"]["BUY"]:>4} sampel ({res["buy_pct"]:5.1f}%)')
    print(f'  🟡 HOLD : {res["counts"]["HOLD"]:>4} sampel ({res["hold_pct"]:5.1f}%)')
    print(f'  🔴 SELL : {res["counts"]["SELL"]:>4} sampel ({res["sell_pct"]:5.1f}%)')

print('  CONTOH REKOMENDASI (5 SAMPEL PERTAMA TEST SET)')

# Re-generate 5 sampel untuk display
for horizon in ['daily', 'weekly']:
    print(f'\n Horizon: {horizon.upper()}')
    if horizon in best_tuned_models:
        mdl = best_tuned_models[horizon][0]
    else:
        mdl = list(BASELINE_MODELS.values())[0]

    X5   = test_df[FEATURES].iloc[:5]
    base = (test_df['price_per_gram'].iloc[:5].values
            if 'price_per_gram' in test_df.columns
            else test_df[FEATURES]['lag_1'].iloc[:5].values)
    deltas = mdl.predict(X5)

    for j, (delta, bp) in enumerate(zip(deltas, base)):
        pp      = bp + delta
        row     = test_df.iloc[j]
        ma7     = row.get('ma_7', np.nan)  if 'ma_7'  in test_df.columns else np.nan
        ma14    = row.get('ma_14', np.nan) if 'ma_14' in test_df.columns else np.nan
        ma_sig, _ = ma_signal(ma7, ma14)
        rec     = generate_recommendation(float(bp), float(pp), horizon=horizon)
        rec     = enrich_recommendation(rec, ma_sig)
        icon    = REC_COLORS[rec['recommendation']]
        print(f'  [{j}] Harga: Rp{bp:>10,.0f} → Pred: Rp{pp:>10,.0f} '
              f'({rec["delta_pct"]:+.2f}%) | MA:{ma_sig:8s} | {icon} {rec["recommendation"]}')

print('\n Semua rekomendasi berhasil ditampilkan!')


  DISTRIBUSI REKOMENDASI PADA TEST SET

  📅 Horizon: DAILY
  🟢 BUY  :   16 sampel ( 40.0%)
  🟡 HOLD :   19 sampel ( 47.5%)
  🔴 SELL :    5 sampel ( 12.5%)

  📅 Horizon: WEEKLY
  🟢 BUY  :    4 sampel ( 10.0%)
  🟡 HOLD :   26 sampel ( 65.0%)
  🔴 SELL :   10 sampel ( 25.0%)
  CONTOH REKOMENDASI (5 SAMPEL PERTAMA TEST SET)

 Horizon: DAILY
  [0] Harga: Rp 3,068,000 → Pred: Rp 3,027,294 (-1.33%) | MA:bullish  | 🟡 HOLD
  [1] Harga: Rp 3,078,000 → Pred: Rp 3,094,387 (+0.53%) | MA:bullish  | 🟢 BUY
  [2] Harga: Rp 3,078,000 → Pred: Rp 3,097,824 (+0.64%) | MA:bullish  | 🟢 BUY
  [3] Harga: Rp 3,080,000 → Pred: Rp 3,083,885 (+0.13%) | MA:bullish  | 🟢 BUY
  [4] Harga: Rp 3,036,000 → Pred: Rp 3,095,868 (+1.97%) | MA:bearish  | 🟢 BUY

 Horizon: WEEKLY
  [0] Harga: Rp 3,068,000 → Pred: Rp 3,028,508 (-1.29%) | MA:bullish  | 🟡 HOLD
  [1] Harga: Rp 3,078,000 → Pred: Rp 3,046,807 (-1.01%) | MA:bullish  | 🟡 HOLD
  [2] Harga: Rp 3,078,000 → Pred: Rp 3,049,393 (-0.93%) | MA:bullish  | 🟡 HOLD
  [3] Harga: Rp 

## 13. Optuna Visualization (disimpan sebagai Artifact)
> Plot optimization history dan parameter importance untuk setiap study

In [24]:
mlflow.set_experiment(EXP_TUNING)

for horizon, study in all_studies.items():
    run_id = best_tuned_run_ids.get(horizon)
    if run_id is None:
        continue

    with mlflow.start_run(run_id=run_id):

        # Optimization history
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f'Optuna Study — {horizon}', fontsize=13)

        trial_nums = [t.number for t in study.trials]
        trial_vals = [t.value  for t in study.trials]
        best_so_far = [min(trial_vals[:i+1]) for i in range(len(trial_vals))]

        axes[0].plot(trial_nums, trial_vals, 'o', alpha=0.5, label='Trial RMSE')
        axes[0].plot(trial_nums, best_so_far, '-r', label='Best so far')
        axes[0].set_title('Optimization History')
        axes[0].set_xlabel('Trial')
        axes[0].set_ylabel('Val RMSE')
        axes[0].legend()

        # Param distribution for numeric params
        param_vals = {}
        for t in study.trials:
            for k, v in t.params.items():
                if isinstance(v, (int, float)):
                    param_vals.setdefault(k, []).append(v)

        if param_vals:
            top_params = sorted(param_vals.items(), key=lambda x: len(x[1]), reverse=True)[:5]
            labels = [p[0] for p in top_params]
            # Normalize for visual
            data = [[v / max(p[1]) for v in p[1]] for p in top_params]
            axes[1].boxplot(data, labels=labels, vert=True)
            axes[1].set_title('Hyperparameter Distribution (normalized)')
            axes[1].tick_params(axis='x', rotation=20)

        fig.tight_layout()
        plot_path = f'optuna_study_{horizon}.png'
        fig.savefig(plot_path, dpi=100)
        plt.close(fig)
        mlflow.log_artifact(plot_path)
        os.remove(plot_path)

        print(f'[{horizon}] Optuna plot logged as artifact')

print('\n🎉 All done! Buka MLflow UI:')
print('   mlflow ui --backend-store-uri file:./mlruns')

[daily] Optuna plot logged as artifact
[weekly] Optuna plot logged as artifact

🎉 All done! Buka MLflow UI:
   mlflow ui --backend-store-uri file:./mlruns


In [25]:
import pickle
import os
import shutil
import mlflow.sklearn
import mlflow.xgboost

PKL_OUTPUT_DIR = './pkl_output'
os.makedirs(PKL_OUTPUT_DIR, exist_ok=True)

TRACKING_URI = 'file:./mlruns'
mlflow.set_tracking_uri(TRACKING_URI)

# Alias yang ingin diexport
ALIASES_TO_EXPORT = ['champion', 'candidate', 'baseline']

exported = []

for horizon in ['daily', 'weekly']:
    reg_name = f'GoldSight-{horizon.capitalize()}-Predictor'

    for alias in ALIASES_TO_EXPORT:
        model_uri = f'models:/{reg_name}@{alias}'

        try:
            # ── Coba load sklearn flavor dulu ──────────────────────────────
            try:
                model = mlflow.sklearn.load_model(model_uri)
                flavor = 'sklearn'
            except Exception:
                model = mlflow.xgboost.load_model(model_uri)
                flavor = 'xgboost'

            model_type = type(model).__name__

            # ── Nama file pkl ───────────────────────────────────────────────
            pkl_filename = f'{alias}_{horizon}_{model_type}.pkl'
            pkl_path     = os.path.join(PKL_OUTPUT_DIR, pkl_filename)

            # ── Simpan ke pkl ───────────────────────────────────────────────
            with open(pkl_path, 'wb') as f:
                pickle.dump(model, f, protocol=pickle.HIGHEST_PROTOCOL)

            size_kb = os.path.getsize(pkl_path) / 1024
            print(f'[{horizon}] @{alias:<10} → {pkl_filename} ({size_kb:.1f} KB)')
            exported.append({
                'horizon':    horizon,
                'alias':      alias,
                'model_type': model_type,
                'flavor':     flavor,
                'pkl_path':   pkl_path,
                'size_kb':    round(size_kb, 1),
            })

        except Exception as e:
            print(f'[{horizon}] @{alias:<10} → tidak tersedia: {e}')

print(f'\nTotal pkl berhasil diexport: {len(exported)}')
print(f'Lokasi: {os.path.abspath(PKL_OUTPUT_DIR)}')


[daily] @champion   → champion_daily_LinearRegression.pkl (0.8 KB)
[daily] @candidate  → candidate_daily_XGBRegressor.pkl (311.7 KB)
[daily] @baseline   → baseline_daily_LinearRegression.pkl (0.8 KB)
[weekly] @champion   → champion_weekly_XGBRegressor.pkl (497.3 KB)
[weekly] @candidate  → candidate_weekly_XGBRegressor.pkl (497.3 KB)
[weekly] @baseline   → baseline_weekly_LinearRegression.pkl (0.8 KB)

Total pkl berhasil diexport: 6
Lokasi: d:\LAPORAN TUGAS\PBL\pkl_output


In [26]:
# ── Verifikasi semua pkl yang diexport ───────────────────────────────────────
import pickle, os
import pandas as pd
import numpy as np

FEATURES = ['weight','lag_1','lag_3','lag_7','ma_7','ma_14',
            'rolling_std_7','momentum','trend_strength','day_of_week']

# Dummy input untuk test predict
dummy = pd.DataFrame([{
    'weight': 1.0, 'lag_1': 3300000.0, 'lag_3': 3280000.0,
    'lag_7': 3250000.0, 'ma_7': 3290000.0, 'ma_14': 3270000.0,
    'rolling_std_7': 15000.0, 'momentum': 50000.0,
    'trend_strength': 20000.0, 'day_of_week': 1
}])[FEATURES]

print('  VERIFIKASI PKL')

pkl_files = [f for f in os.listdir('./pkl_output') if f.endswith('.pkl')]

for fname in sorted(pkl_files):
    path = os.path.join('./pkl_output', fname)
    try:
        with open(path, 'rb') as f:
            model = pickle.load(f)
        y = model.predict(dummy)
        size_kb = os.path.getsize(path) / 1024
        print(f'  {fname}')
        print(f'     Tipe  : {type(model).__name__}')
        print(f'     Output: Rp {y[0]:+,.0f}/gram (delta prediksi)')
        print(f'     Size  : {size_kb:.1f} KB')
    except Exception as e:
        print(f' {fname} → {e}')


print(f'  Total: {len(pkl_files)} file pkl')


  VERIFIKASI PKL
  baseline_daily_LinearRegression.pkl
     Tipe  : LinearRegression
     Output: Rp +28,627/gram (delta prediksi)
     Size  : 0.8 KB
  baseline_weekly_LinearRegression.pkl
     Tipe  : LinearRegression
     Output: Rp -144,290/gram (delta prediksi)
     Size  : 0.8 KB
  candidate_daily_XGBRegressor.pkl
     Tipe  : XGBRegressor
     Output: Rp +12,052/gram (delta prediksi)
     Size  : 311.7 KB
  candidate_weekly_XGBRegressor.pkl
     Tipe  : XGBRegressor
     Output: Rp -106,886/gram (delta prediksi)
     Size  : 497.3 KB
  champion_daily_LinearRegression.pkl
     Tipe  : LinearRegression
     Output: Rp +28,627/gram (delta prediksi)
     Size  : 0.8 KB
  champion_weekly_XGBRegressor.pkl
     Tipe  : XGBRegressor
     Output: Rp -106,886/gram (delta prediksi)
     Size  : 497.3 KB
  Total: 6 file pkl
